In [1]:
import pymongo
from sympy.codegen.ast import break_

client = pymongo.MongoClient("mongodb://localhost:27017/")
langweb_db = client['langweb']
verbs_collection = langweb_db['verbs']
games_collection = langweb_db['games']

In [2]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
import unicodedata
options=Options()
driver = webdriver.Chrome()
verbs = []
def get_verbs(url):
    driver.get(url)
    WebDriverWait(driver, 1)
    driver.find_element("xpath" ,"/html/body/div[1]/div/div/div/div/div/div[2]/button[2]").send_keys(Keys.ENTER)
    frame_verb=driver.find_element("xpath" ,"//*[@id='form1']/div[3]/div/div[1]/div[4]/ol")
    
    elements = frame_verb.find_elements("tag name", "li")
    print(len(elements))
    verbs.extend([element.text for element in elements])
    WebDriverWait(driver, 1)
    driver.close()
interval="1-2000"
url = f"https://conjugator.reverso.net/index-german-{interval}.html"
get_verbs(url)
    


2000


In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
def create_dataset(verb):
    verb_obj = {}
    def openpage():
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, "/html/body/nav/div/div[1]/form/table/tbody/tr/td[1]/div/input[1]")))
        driver.find_element(By.XPATH, "/html/body/nav/div/div[1]/form/table/tbody/tr/td[1]/div/input[1]").send_keys(verb)
        driver.find_element(By.XPATH, "/html/body/nav/div/div[1]/form/table/tbody/tr/td[2]/button").click()
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH,"//*[@id='vStckKrz']/div[2]/p[1]/span")))
    
    def verbinfo():
        meaning = driver.find_element(By.XPATH,"//*[@id='vStckKrz']/div[2]/p[1]/span").text
        meaning = ", ".join(meaning.split(", ")[:3])
        otherinfo = driver.find_element(By.XPATH, "//*[@id='vVdBx']/p").text

        if otherinfo.split("·")[1].strip() == "unregelmäßig":
            regularity = False
        else: regularity = True

        root_change_info = driver.find_element(By.XPATH, "//*[@id='vStckKrz']/p[2]").text
        if root_change_info.strip():
            root_change = True
        else: root_change = False

        if otherinfo.split("·")[-1].strip() == "untrennbar":
            separability = False
        elif otherinfo.split("·")[-1].strip() == "trennbar": 
            separability = True
        else: separability = None
        info_dict = {"verb": verb,
                "meaning": meaning,
                "regular": regularity,
                "root_change": root_change,
                "separable": separability,
                "reflexive": False,
        }
        return info_dict
    
    def conjugations():
        parent = driver.find_element(By.CLASS_NAME, "rAbschnitt")
        indikativ_parent = parent.find_element(By.XPATH, "div/section[3]/div[2]")
        times = indikativ_parent.find_elements(By.CLASS_NAME, "vTbl")
        pronouns = ["ich","du","er/sie/es","wir","ihr", "sie/Sie"]
        indikativ_dict={}
        for time in times:
            tense= time.find_element(By.TAG_NAME,"h3").text
            tense = tense.replace(".","")
            indikativ_dict[tense]={}
            for pronoun in pronouns:
                conjugation = time.find_element(By.XPATH, f"table/tbody/tr[{pronouns.index(pronoun)+1}]").text
                conjugation = ''.join(c for c in conjugation if unicodedata.category(c) != 'No' and c not in '()')
                conjugation = " ".join(conjugation.split(" ")[1:])
                indikativ_dict[tense][pronoun] = conjugation
        verb_obj["Indikativ"] = indikativ_dict
                

        konjunktiv_parent = parent.find_element(By.XPATH, "div/section[4]/div[2]")
        times = konjunktiv_parent.find_elements(By.CLASS_NAME, "vTbl")
        pronouns = ["ich","du","er/sie/es","wir","ihr", "sie/Sie"]
        konjunktiv_dict={}
        for time in times:
            tense= time.find_element(By.TAG_NAME,"h3").text
            tense = tense.replace(".","")
            if tense.startswith("Konj "): tense = tense.replace("Konj ", "")
            konjunktiv_dict[tense]={}
            for pronoun in pronouns:
                conjugation = time.find_element(By.XPATH, f"table/tbody/tr[{pronouns.index(pronoun)+1}]").text 
                conjugation = ''.join(c for c in conjugation if unicodedata.category(c) != 'No' and c not in '()')
                conjugation = " ".join(conjugation.split(" ")[1:])
                konjunktiv_dict[tense][pronoun] = conjugation
        verb_obj["Konjunktiv"] = konjunktiv_dict
                
                
        imperativ_parent = parent.find_element(By.XPATH, "div/section[6]/div[2]")
        times = imperativ_parent.find_elements(By.CLASS_NAME, "vTbl")
        pronouns = ["du","wir","ihr", "Sie"]
        imperativ_dict={}
        for time in times:
            tense= time.find_element(By.TAG_NAME,"h3").text
            imperativ_dict[tense]={}
            for pronoun in pronouns:
                conjugation = time.find_element(By.XPATH, f"table/tbody/tr[{pronouns.index(pronoun)+1}]").text
                conjugation = ''.join(c for c in conjugation if unicodedata.category(c) != 'No' and c not in '()')
                conjugation = conjugation.replace(" "+pronoun,"")
                imperativ_dict[tense][pronoun] = conjugation
        verb_obj["Imperativ"] = imperativ_dict

    
    openpage()
    try:
        info_dict = verbinfo()
        verb_obj= info_dict
        conjugations()
        verbs_collection.insert_one(verb_obj)
        print("data added")
    except:
        url = driver.current_url
        print(url)
        try:
            div = driver.find_element(By.XPATH, "/html/body/article/div[1]")
            child = div.find_elements(By.XPATH, "./a")[1]
            href_value = child.get_attribute("href")
            if href_value.startswith("https://www.verbformen.de/konjugation"):
                driver.get(href_value)
                info_dict = verbinfo()
                verb_obj= info_dict
                conjugations()
                verbs_collection.insert_one(verb_obj)
                print("data added")
        except:
            print(f"No verb found for {verb}.")
    
    

    if verb_obj["separable"] != None:
        div = driver.find_element(By.XPATH, "/html/body/article/div[1]")
        children = div.find_elements(By.XPATH, "./a")
        number_of_children = len(children)
        for i in range(1, number_of_children):
            div = driver.find_element(By.XPATH, "/html/body/article/div[1]")
            child = div.find_elements(By.XPATH, "./a")[i]
            href_value = child.get_attribute("href")
            img_value = child.find_element(By.TAG_NAME, "img").get_attribute("src")
            if href_value.startswith("https://www.verbformen.de/konjugation") and img_value == "https://www.verbformen.de/unselected.svg":
                driver.get(href_value)
                WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH,"//*[@id='vStckKrz']/div[2]/p[1]/span")))
                info_dict = verbinfo()
                verb_obj= info_dict
                conjugations()
                WebDriverWait(driver, 10)
                verbs_collection.insert_one(verb_obj)
                print("data added")
            
    

    WebDriverWait(driver, 3)
    driver.find_element(By.XPATH, "/html/body/nav/div/div[1]/form/table/tbody/tr/td[1]/div/input[1]").clear()
    # print(verb_obj)
    return verb_obj
    
driver = webdriver.Chrome()
driver.get("https://www.verbformen.de/")
WebDriverWait(driver, 1)
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH,"/html/body/div[6]/iframe")))
frame_cookies= driver.find_element(By.XPATH, "/html/body/div[6]/iframe")
driver.switch_to.frame(frame_cookies)
WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH,"/html/body/div/div[2]/div[5]/button"))).click()
driver.switch_to.default_content()

WebDriverWait(driver, 1)
options.page_load_strategy = 'eager'
for verb in verbs[:550]:
    try:
        verb_obj = create_dataset(verb)
    except Exception as e:
        print(f"unsuccessful for {verb}")

driver.close()

In [ ]:
# def get_conjugations(verb):
#     driver.find_element("xpath", "/html/body/div[2]/div[1]/div/div[1]/div/form/div[3]/div/div[1]/div[2]/div[1]/div/div[1]/input").send_keys(verb)
#     driver.find_element("xpath", "/html/body/div[2]/div[1]/div/div[1]/div/form/div[3]/div/div[1]/div[2]/div[1]/div/a").send_keys(Keys.ENTER)
#     options.page_load_strategy = 'eager'
#     WebDriverWait(driver, 1)
#     
# driver = webdriver.Chrome()
# driver.get("https://conjugator.reverso.net/conjugation-german.html")
# WebDriverWait(driver, 1)
# driver.find_element("xpath","/html/body/div[1]/div/div/div/div/div/span").click()
# for verb in verbs[:5]:
#     print(verb)
# 
#     get_conjugations(verb)

In [6]:
german_words = [
    "die Erinnerung", "der Zweifel", "die Gelegenheit", "der Zufall", "die Entscheidung",
    "das Abenteuer", "die Erfahrung", "die Verantwortung", "der Gedanke", "der Traum",
    "die Hoffnung", "die Freiheit", "die Angst", "das Vertrauen", "die Zukunft",
    "die Vergangenheit", "die Gegenwart", "der Augenblick", "die Wirklichkeit", "der Schatten",
    "das Licht", "die Stille", "das Geräusch", "die Bewegung", "der Regen",
    "der Sturm", "der Himmel", "die Erde", "der Fluss", "der Berg",
    "der Wald", "das Meer", "die Insel", "der Stern", "der Mond",
    "die Sonne", "die Blume", "der Baum", "das Blatt", "der Stein",
    "der Weg", "die Straße", "die Brücke", "das Haus", "die Tür",
    "das Fenster", "das Zimmer", "der Tisch", "der Stuhl", "das Bett",
    "der Spiegel", "die Wand", "der Boden", "das Dach", "die Treppe",
    "der Garten", "das Schloss", "das Dorf", "die Stadt", "das Land",
    "die Grenze", "der Pass", "die Reise", "der Zug", "das Auto",
    "das Flugzeug", "das Schiff", "der Hafen", "der Markt", "das Geschäft",
    "das Geld", "der Preis", "die Arbeit", "der Beruf", "die Aufgabe",
    "der Chef", "der Kollege", "das Büro", "die Firma", "die Maschine",
    "der Computer", "das Programm", "der Bildschirm", "die Tastatur", "die Sprache",
    "das Wort", "der Satz", "der Text", "das Buch", "die Geschichte",
    "das Kapitel", "der Roman", "der Autor", "die Meinung", "das Thema",
    "der Fehler", "die Lösung", "der Versuch", "die Möglichkeit", "der Erfolg",
    "der Misserfolg", "der Plan", "das Ziel", "die Richtung", "die Regel",
    "das Gesetz", "der Staat", "die Gesellschaft", "die Politik", "der Krieg",
    "der Frieden", "die Kultur", "die Kunst", "das Bild", "das Museum",
    "die Musik", "das Lied", "die Melodie", "der Klang", "der Film",
    "die Szene", "der Schauspieler", "die Rolle", "das Theater", "die Bühne",
    "die Kamera", "das Publikum", "der Applaus", "die Liebe", "der Hass",
    "die Freude", "der Schmerz", "das Glück", "das Unglück", "die Überraschung",
    "die Entscheidung", "die Meinung", "der Gedanke", "der Traum", "der Wunsch",
    "die Sehnsucht", "die Stimme", "der Geruch", "der Geschmack", "das Gefühl",
    "der Körper", "der Kopf", "das Herz", "die Hand", "der Blick",
    "denken", "fühlen", "träumen", "hoffen", "glauben",
    "zweifeln", "entscheiden", "vergessen", "erinnern", "suchen",
    "finden", "verlieren", "gewinnen", "warten", "beginnen",
    "enden", "bleiben", "gehen", "kommen", "laufen",
    "rennen", "fahren", "fliegen", "schwimmen", "stehen",
    "sitzen", "liegen", "sehen", "hören", "sprechen",
    "sagen", "erzählen", "fragen", "antworten", "schreiben",
    "lesen", "verstehen", "erklären", "benutzen", "schaffen",
    "arbeiten", "leben", "lieben", "hassen", "lachen",
    "weinen", "spielen", "lernen", "verändern", "wachsen"
]


In [7]:
german_nouns_20q = [
    # 🏠 Dinge & Alltagsobjekte
    "das Haus", "die Wohnung", "das Zimmer", "der Tisch", "der Stuhl", "das Bett",
    "der Schrank", "das Regal", "die Lampe", "der Spiegel", "die Uhr", "der Teppich",
    "das Fenster", "die Tür", "das Dach", "die Wand", "der Boden", "die Treppe",
    "die Kerze", "der Fernseher", "das Radio", "der Computer", "das Handy", "das Tablet",
    "die Tastatur", "die Maus", "der Bildschirm", "der Drucker", "die Kamera", "das Mikrofon",
    "der Kühlschrank", "der Ofen", "die Waschmaschine", "der Trockner", "der Toaster", "die Pfanne",
    "der Topf", "das Messer", "die Gabel", "der Löffel", "der Teller", "das Glas",
    "die Tasse", "die Flasche", "der Becher", "die Schüssel", "der Besen", "der Staubsauger",
    "das Buch", "die Zeitung", "das Heft", "der Stift", "der Zettel", "das Papier",
    "die Tasche", "der Rucksack", "der Koffer", "der Schlüssel", "das Portemonnaie", "die Brille",
    "die Sonnenbrille", "die Uhr", "das Armband", "der Ring", "die Kette", "die Jacke",
    "das Hemd", "die Hose", "der Rock", "das Kleid", "die Bluse", "die Schuhe",
    "die Socken", "der Mantel", "die Mütze", "der Hut", "der Schal", "die Handschuhe",

    # 🧍‍♀️ Berufe & Menschen
    "der Lehrer/die Lehrerin", "der Arzt/die Ärztin", "der Koch/die Köchin",
    "der Kellner/die Kellnerin", "der Verkäufer/die Verkäuferin", "der Ingenieur/die Ingenieurin",
    "der Polizist/die Polizistin", "der Feuerwehrmann", "die Krankenschwester", "der Mechaniker", "der Elektriker",
    "der Musiker", "die Sängerin", "der Schauspieler/die Schauspielerin", "der Pilot/die Pilotin",
    "der Fahrer/die Fahrerin", "der Gärtner", "die Gärtnerin", "der Maler", "die Architektin",
    "der Student/die Studentin", "der Professor", "die Forscherin", "der Bauer/die Bäuerin",
    "der Bäcker", "die Friseurin", "der Journalist", "die Reporterin", "der Programmierer", "die Designerin",

    # 🐶 Tiere
    "der Hund", "die Katze", "der Vogel", "der Fisch", "das Pferd", "die Maus",
    "der Hase", "der Elefant", "die Giraffe", "der Löwe", "der Tiger", "der Affe",
    "das Känguru", "der Bär", "der Fuchs", "der Wolf", "das Schaf", "die Kuh",
    "das Schwein", "das Huhn", "die Ente", "der Hahn", "der Adler", "die Eule",
    "der Pinguin", "der Delfin", "der Wal", "der Hai", "das Krokodil", "die Schildkröte",
    "die Biene", "die Fliege", "die Ameise", "der Schmetterling", "die Spinne", "die Schlange",

    # 🍎 Essen & Trinken
    "das Brot", "die Butter", "der Käse", "die Milch", "der Kaffee", "der Tee",
    "das Wasser", "die Suppe", "das Ei", "das Salz", "der Zucker", "der Reis",
    "die Nudeln", "die Kartoffel", "das Gemüse", "das Obst", "der Apfel", "die Banane",
    "die Orange", "die Zitrone", "die Erdbeere", "die Tomate", "die Gurke", "die Karotte",
    "die Zwiebel", "der Knoblauch", "der Salat", "die Pizza", "der Kuchen", "das Eis",
    "die Schokolade", "der Keks", "das Bonbon", "der Honig", "der Joghurt", "das Bier",
    "der Wein", "der Saft", "die Marmelade", "das Frühstück", "das Mittagessen", "das Abendessen",

    # 🌍 Orte
    "die Stadt", "das Dorf", "das Land", "die Straße", "die Brücke", "der Park",
    "der Garten", "die Schule", "die Universität", "das Krankenhaus", "die Kirche", "das Kino",
    "das Theater", "das Museum", "die Bibliothek", "der Bahnhof", "der Flughafen", "der Hafen",
    "das Restaurant", "das Café", "der Supermarkt", "das Geschäft", "die Bank", "die Post",
    "das Stadion", "die Fabrik", "das Büro", "der Markt", "die Polizei", "das Schwimmbad",
    "das Hotel", "die Wohnung", "das Zimmer", "der Balkon", "die Garage", "der Keller",

    # 🌳 Natur & Umwelt
    "der Baum", "die Blume", "das Blatt", "die Wiese", "der Wald", "der Fluss",
    "der See", "das Meer", "der Berg", "das Tal", "die Insel", "der Strand",
    "der Stein", "der Sand", "der Regen", "der Schnee", "die Wolke", "der Himmel",
    "die Sonne", "der Mond", "der Stern", "die Luft", "das Feuer", "das Wasser",
    "die Erde", "der Wind", "der Sturm", "der Nebel", "der Donner", "der Blitz",

    # 🚗 Verkehr & Transport
    "das Auto", "das Fahrrad", "der Bus", "die Bahn", "der Zug", "die U-Bahn",
    "das Flugzeug", "das Schiff", "das Motorrad", "der Lastwagen", "das Taxi", "die Ampel",
    "die Kreuzung", "die Straße", "die Brücke", "der Tunnel", "der Fahrer", "die Tankstelle",
    "das Ticket", "der Flughafen", "die Haltestelle", "das Fahrrad", "der Helm", "die Karte",

    # 🧠 Abstrakte / Sonstige (aber noch erratbar)
    "das Spiel", "der Sport", "die Musik", "das Lied", "das Instrument", "der Film",
    "das Foto", "die Farbe", "die Zahl", "die Sprache", "das Wort", "die Frage",
    "die Antwort", "der Name", "der Traum", "das Jahr", "der Monat", "der Tag",
    "die Stunde", "die Minute", "die Sekunde", "die Zeit", "der Körper", "der Kopf",
    "das Herz", "die Hand", "der Fuß", "das Auge", "das Ohr", "die Nase",
    "der Mund", "die Stimme", "der Zahn", "das Blut", "die Haut", "das Haar",

    # 💡 Technische & moderne Begriffe
    "das Internet", "die E-Mail", "die Website", "die App", "der Code", "das Passwort",
    "das Konto", "die Datei", "der Ordner", "die Nachricht", "das Video", "das Foto",
    "das Programm", "die Software", "die Hardware", "der Roboter", "das Kabel", "der Lautsprecher",
    "die Batterie", "das Ladegerät", "die Kamera", "das Mikrofon", "der Bildschirm", "die Tastatur"
]


In [8]:
was_waere_wenn_fragen = [
    "Was wäre, wenn du einen Tag lang unsichtbar wärst?",
    "Was wäre, wenn du in einem anderen Land aufwachen würdest?",
    "Was wäre, wenn du plötzlich fliegen könntest?",
    "Was wäre, wenn du keine Angst vor irgendetwas hättest?",
    "Was wäre, wenn du in die Zukunft sehen könntest?",
    "Was wäre, wenn du die Gedanken anderer Menschen hören könntest?",
    "Was wäre, wenn du nie wieder schlafen müsstest?",
    "Was wäre, wenn du deinen Traumjob hättest?",
    "Was wäre, wenn du eine Million Euro gewinnen würdest?",
    "Was wäre, wenn du für einen Tag berühmt wärst?",
    "Was wäre, wenn du dein Lieblingsessen nie wieder essen dürftest?",
    "Was wäre, wenn du einen Tag als Tier leben könntest?",
    "Was wäre, wenn du morgen in einer anderen Zeit leben würdest – z. B. im Mittelalter?",
    "Was wäre, wenn du nur noch eine Sprache sprechen könntest?",
    "Was wäre, wenn du ein Jahr ohne Internet leben müsstest?",
    "Was wäre, wenn du den Beruf deiner Eltern übernehmen müsstest?",
    "Was wäre, wenn du dich plötzlich in einem fremden Körper wiederfinden würdest?",
    "Was wäre, wenn du dein Lieblingslied nie wieder hören dürftest?",
    "Was wäre, wenn du die Welt für einen Tag regieren würdest?",
    "Was wäre, wenn du alles vergessen würdest, was du heute weißt?",
    "Was wäre, wenn du in einem anderen Jahrhundert geboren wärst?",
    "Was wäre, wenn du nie erwachsen werden würdest?",
    "Was wäre, wenn du alles essen könntest, ohne zuzunehmen?",
    "Was wäre, wenn du jeden Tag denselben Traum hättest?",
    "Was wäre, wenn du morgen im Lotto gewinnen würdest?",
    "Was wäre, wenn du der letzte Mensch auf der Erde wärst?",
    "Was wäre, wenn du dich teleportieren könntest?",
    "Was wäre, wenn du einen Tag als dein Lieblingsschauspieler leben könntest?",
    "Was wäre, wenn du in einem anderen Land geboren wärst?",
    "Was wäre, wenn du die Zeit anhalten könntest?",
    "Was wäre, wenn du dich in dein Lieblingsbuch hineinversetzen könntest?",
    "Was wäre, wenn du nur noch flüstern dürftest?",
    "Was wäre, wenn du ein Jahr lang keine Musik hören dürftest?",
    "Was wäre, wenn du dich für immer in ein Tier verwandeln würdest?",
    "Was wäre, wenn du dein Handy für einen Monat nicht benutzen dürftest?",
    "Was wäre, wenn du dich jeden Tag an einem neuen Ort wiederfinden würdest?",
    "Was wäre, wenn du in einem anderen Beruf arbeiten müsstest?",
    "Was wäre, wenn du plötzlich berühmt wärst, aber niemand dich mochte?",
    "Was wäre, wenn du für einen Tag das Wetter kontrollieren könntest?",
    "Was wäre, wenn du nie wieder lügen könntest?",
    "Was wäre, wenn du morgen aufwachst und alle Menschen verschwunden sind?",
    "Was wäre, wenn du deinen besten Freund verlieren würdest?",
    "Was wäre, wenn du eine Woche lang alles tun könntest, was du willst?",
    "Was wäre, wenn du in einem Film leben müsstest?",
    "Was wäre, wenn du ein Jahr lang reisen müsstest, ohne nach Hause zu kommen?",
    "Was wäre, wenn du ein Geheimnis erfahren würdest, das niemand wissen darf?",
    "Was wäre, wenn du plötzlich reich, aber sehr einsam wärst?",
    "Was wäre, wenn du ein Instrument perfekt spielen könntest?",
    "Was wäre, wenn du für immer in deinem Lieblingsalter bleiben könntest?",
    "Was wäre, wenn du einen Tag lang dein eigenes Leben aus der Ferne beobachten würdest?"
]


In [9]:
situations = [
    # 🏠 Alltag & Soziales
    "Im Supermarkt: Eine Person sucht ein Produkt, die andere arbeitet dort.",
    "Im Restaurant: Ein Gast beschwert sich über das Essen.",
    "In der Bäckerei: Der Kunde will etwas kaufen, aber hat kein Bargeld.",
    "Beim Arzt: Ein Patient erklärt seine Symptome.",
    "In der Schule: Ein Schüler hat seine Hausaufgaben nicht gemacht.",
    "Im Zug: Zwei Reisende streiten sich über den Sitzplatz.",
    "Im Hotel: Der Gast ist mit dem Zimmer nicht zufrieden.",
    "Im Kino: Jemand spricht zu laut während des Films.",
    "In der Bibliothek: Eine Person sucht ein bestimmtes Buch.",
    "Im Fitnessstudio: Der Trainer gibt Tipps, der Schüler ist faul.",

    # 💼 Arbeit & Büro
    "Im Büro: Ein Kollege kommt immer zu spät.",
    "Beim Bewerbungsgespräch: Der Bewerber ist nervös.",
    "Bei einer Präsentation: Der Computer funktioniert nicht.",
    "In einer Besprechung: Zwei Mitarbeiter sind sich nicht einig.",
    "Am ersten Arbeitstag: Ein neuer Mitarbeiter stellt sich vor.",
    "Bei einer Kündigung: Der Chef erklärt die Entscheidung.",
    "Bei einem Projekt: Die Gruppe diskutiert über die beste Idee.",
    "Im Homeoffice: Der Internetanschluss fällt aus.",
    "Auf einer Geschäftsreise: Das Gepäck ist verloren gegangen.",
    "In einer Firma: Ein Kunde ruft wegen einer Beschwerde an.",

    # 👪 Familie & Freunde
    "Zu Hause: Eltern und Kind streiten über das Handy.",
    "Beim Abendessen: Jemand hat ein Geheimnis.",
    "Beim Geburtstag: Ein Geschenk gefällt nicht.",
    "In einer WG: Ein Mitbewohner macht nie sauber.",
    "Bei einem Streit: Zwei Freunde haben unterschiedliche Meinungen.",
    "Am Wochenende: Freunde planen einen Ausflug.",
    "Im Urlaub: Das Hotelzimmer ist anders als erwartet.",
    "Beim Telefonat: Eine Mutter ruft ihr Kind an.",
    "Beim Picknick: Es beginnt plötzlich zu regnen.",
    "Beim Umzug: Freunde helfen beim Tragen der Möbel.",

    # 🚨 Unerwartete Situationen
    "Ein Auto hat eine Panne mitten auf der Straße.",
    "Jemand hat seine Tasche verloren.",
    "Ein Hund läuft plötzlich weg.",
    "Ein Handy fällt ins Wasser.",
    "Im Park: Ein Kind hat sich verlaufen.",
    "Im Zug: Eine Person hat kein Ticket.",
    "Im Restaurant: Die Rechnung stimmt nicht.",
    "Im Hotel: Das Gepäck wurde vertauscht.",
    "In der Stadt: Ein Tourist fragt nach dem Weg.",
    "Im Geschäft: Der Kunde merkt, dass er kein Geld dabei hat.",

    # 🌍 Reisen & Freizeit
    "Am Flughafen: Der Flug ist verspätet.",
    "Am Strand: Jemand verliert seine Sonnenbrille im Meer.",
    "In den Bergen: Wanderer treffen auf schlechtes Wetter.",
    "Im Museum: Ein Kind berührt ein Kunstwerk.",
    "Auf dem Festival: Zwei Leute verlieren ihre Freunde.",
    "Beim Camping: Das Zelt ist kaputt.",
    "Auf einer Party: Jemand verschüttet ein Getränk.",
    "Im Café: Das letzte Stück Kuchen ist heiß begehrt.",
    "In der U-Bahn: Ein Musiker spielt laut Musik.",
    "Auf dem Markt: Der Verkäufer bietet ein besonderes Produkt an.",

    # 💡 Fantasie & Humor
    "In der Zukunft: Roboter arbeiten als Lehrer.",
    "Auf einem anderen Planeten: Zwei Aliens reden über die Erde.",
    "In einem Märchenwald: Eine Hexe trifft einen Ritter.",
    "In einer Zeitmaschine: Die Reisenden landen im Jahr 1800.",
    "Im Traum: Alles ist plötzlich auf dem Kopf.",
    "Im Zoo: Die Tiere sprechen plötzlich miteinander.",
    "In der Schule der Superhelden: Jemand hat eine seltsame Superkraft.",
    "In einer Welt ohne Strom: Menschen müssen kreativ werden.",
    "In einem Computerspiel: Eine Figur wird lebendig.",
    "In der Zukunft: Menschen kommunizieren nur noch mit Emojis."
]


In [13]:
games_collection.insert_one({"game1":german_words, "game3":was_waere_wenn_fragen,"game4": situations,"game5": german_nouns_20q})

InsertOneResult(ObjectId('690b17e07217de0e87b640f2'), acknowledged=True)

In [15]:
games = {"game1":german_words, "game3":was_waere_wenn_fragen,"game4": situations,"game5": german_nouns_20q}
import json
with open('../games.json', 'w') as f:
    json.dump(games, f)
